In [1]:
# imports
import copy
%matplotlib inline
from matplotlib import pyplot as plt
import numpy as np
import os
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch import Tensor
import torch.nn as nn
from typing import List, Optional

In [2]:
cwd = os.getcwd()

x_train = pd.read_csv(os.path.join(cwd,'X_train.csv'), index_col=0, sep=',')
x_train.columns.name = 'date'

y_train = pd.read_csv(os.path.join(cwd,'Y_train.csv'), index_col=0, sep=',')
y_train.columns.name = 'date'

In [3]:
x_np = x_train.to_numpy().T
x_np = np.stack([np.flip(x_np[i:i+250], axis=0).T for i in range(len(x_np)-250)])
x_np.shape
x_torch = torch.tensor(x_np)
x_torch.shape
y_np = y_train.to_numpy().T
y_np.shape
y_torch = torch.tensor(y_np)
y_torch.shape

torch.Size([504, 50])

In [4]:
best_metric = {}
best_model = {}
history = {}

In [5]:
def metric_torch(y_pred: Tensor, y_true: Tensor) -> Tensor:
    """Stock returns metric implemented in `torch`.
    
    Parameters
    ----------
    y_pred : torch.Tensor
        tensor of predicted stock returns of shape (m, N).
    y_true : torch.Tensor
        tensor of actual stock returns of shape (m, N).
    
    Returns
    -------
    torch.Tensor
        the average normalised inner product bewteen predicted and actual stock returns.
    """
    y_pred = y_pred.div(y_pred.norm(keepdim=True, dim=1))
    y_true = y_true.div(y_true.norm(keepdim=True, dim=1))
    return torch.einsum('ti,ti->t', y_pred, y_true).mean()

In [6]:
import torch
import torch.nn as nn
from torch import Tensor
import numpy as np


class QRRecencyModel(nn.Module):
    """
    Learn an unconstrained B in R^{D x F}, then build
        A_pre = diag(w_decay) @ B
        A = Q from QR(A_pre)   (orthonormal columns)
    and use A as the factor-loading matrix.

    This "relaxes" orthogonality during optimisation (we work in B-space),
    and enforces it via a QR projection each forward pass.
    """

    def __init__(self, D: int = 250, F: int = 10,
                 gamma: float = 0.02,
                 learn_gamma: bool = False) -> None:
        super().__init__()
        self.D = D
        self.F = F

        # Unconstrained parameter B (D x F)
        self.B = nn.Parameter(
            0.01 * torch.randn(D, F, dtype=torch.float64)
        )

        # Beta parameters (F,)
        self.beta = nn.Parameter(
            torch.randn(F, dtype=torch.float64)
        )

        # Recency decay: w_i ~ exp(-gamma * (D-1-i))
        # i=0: far past; i=D-1: most recent
        w = np.exp(-gamma * (np.arange(D)[::-1]))
        w = w / w.max()  # normalise max to 1 for stability
        w_torch = torch.tensor(w, dtype=torch.float64)

        if learn_gamma:
            # If you want to learn decay shape, make it a parameter
            self.w_decay = nn.Parameter(w_torch)
        else:
            # Otherwise keep as a buffer (fixed prior)
            self.register_buffer("w_decay", w_torch)

    @property
    def A(self) -> Tensor:
        """
        Build orthonormal A from B via recency-weighted QR:

            A_pre = diag(w_decay) @ B
            Q, R = qr(A_pre)
            A = Q[:, :F]
        """
        # Row-wise multiply by decay: shape (D, F)
        A_pre = self.B * self.w_decay.view(self.D, 1)

        # QR decomposition; Q has orthonormal columns
        # mode="reduced" gives Q of shape (D, F) when A_pre is D x F
        Q, _ = torch.linalg.qr(A_pre, mode="reduced")

        # In practice Q is already D x F, but we slice for clarity
        return Q[:, :self.F]

    def factors(self, x: Tensor) -> Tensor:
        """
        x: shape (m, N, D)
        A: shape (D, F)
        returns: (m, N, F)
        """
        return torch.matmul(x, self.A)  # uses broadcasting matmul

    def forward(self, x: Tensor) -> Tensor:
        """
        x: shape (m, N, D)
        beta: shape (F,)
        returns: y_pred shape (m, N)
        """
        F_t = self.factors(x)    # (m, N, F)
        y_pred = torch.matmul(F_t, self.beta)  # (m, N)
        return y_pred


In [7]:
number_of_epochs = 400  # you can tune this
best_metric['qr'] = -1
best_model['qr'] = None
history['qr'] = {'metric': []}

torch.manual_seed(1234)

model_qr = QRRecencyModel(D=250, F=10, gamma=0.02, learn_gamma=False)
optimizer = torch.optim.Adam(model_qr.parameters(), lr=1e-3)

model_qr.train(True)

for epoch in range(number_of_epochs):
    optimizer.zero_grad()

    y_pred = model_qr(x_torch)          # (m, N)
    m_val = metric_torch(y_pred, y_torch)
    loss = -m_val                       # maximise metric

    loss.backward()
    optimizer.step()

    m_float = m_val.item()
    history['qr']['metric'].append(m_float)

    if m_float > best_metric['qr']:
        best_metric['qr'] = m_float
        best_model['qr'] = model_qr
        best_model['qr'].train(False)

    if epoch % 20 == 0:
        print(f"[QR | Epoch {epoch:3d} | Metric: {m_float: .5f}]")

print(f"Best QR metric: {best_metric['qr']: .10f}")


[QR | Epoch   0 | Metric: -0.00392]
[QR | Epoch  20 | Metric:  0.09528]
[QR | Epoch  40 | Metric:  0.11507]
[QR | Epoch  60 | Metric:  0.13161]
[QR | Epoch  80 | Metric:  0.13097]
[QR | Epoch 100 | Metric:  0.11699]
[QR | Epoch 120 | Metric:  0.14506]
[QR | Epoch 140 | Metric:  0.14654]
[QR | Epoch 160 | Metric:  0.14663]
[QR | Epoch 180 | Metric:  0.14626]
[QR | Epoch 200 | Metric:  0.14575]
[QR | Epoch 220 | Metric:  0.14083]
[QR | Epoch 240 | Metric:  0.14269]
[QR | Epoch 260 | Metric:  0.14509]
[QR | Epoch 280 | Metric:  0.14596]
[QR | Epoch 300 | Metric:  0.14683]
[QR | Epoch 320 | Metric:  0.14693]
[QR | Epoch 340 | Metric:  0.14714]
[QR | Epoch 360 | Metric:  0.14772]
[QR | Epoch 380 | Metric:  0.14852]
Best QR metric:  0.1486417544


In [8]:
with torch.no_grad():
    A_torch = best_model['qr'].A          # (250, 10)
    beta_torch = best_model['qr'].beta    # (10,)

A = A_torch.detach().cpu().numpy()
beta = beta_torch.detach().cpu().numpy()

D, F = A.shape  # 250, 10

# Stack A column by column: [A1; A2; ...; A10]
stacked_A = np.concatenate([A[:, j] for j in range(F)], axis=0)

# Final submission vector: [A1; ...; A10; beta]
output_vector = np.concatenate([stacked_A, beta], axis=0)




In [11]:
# to check the orthonormality constraints as in the metric:
    
def checkOrthonormality(A): 
    
    bool = True
    D, F = A.shape   
    Error = pd.DataFrame(A.T @ A - np.eye(F)).abs()
    
    if any(Error.unstack() > 1e-6):
        bool = False
     
    return bool

checkOrthonormality(A)

True

In [13]:
def parametersTransform(A, beta, D=250, F=10):
    
    if A.shape != (D, F):
        print('A has not the good shape')
        return
    
    if beta.shape[0] != F:
        print('beta has not the good shape')
        return        
    
    output = np.hstack( (np.hstack([A.T, beta.reshape((F, 1))])).T )
    
    return output



output = parametersTransform(A, beta)
pd.DataFrame(output).to_csv('submissionqr.csv')

In [9]:
# Save in the required format
df_out = pd.DataFrame({"0": output_vector})
df_out.to_csv("model_parameters_qr.csv")